# 🚦 Entrenamiento de Modelo Detector para Semáforo Inteligente (Entorno Local Windows / Colab)
### Arquitectura: SSD MobileNet V2 FPN-Lite 320x320 (TensorFlow 2 Object Detection API)

> **Nota de entorno:**
> Todas las celdas de este notebook incluyen resolución automática de rutas (`PROJECT_ROOT`, `MODELS_WORK_DIR`, `IMAGES_DIR`, `ANNOTATIONS_DIR`, etc.), por lo que funcionan sin errores de rutas (`NameError` o `FileNotFoundError`) tanto ejecutando en orden como ejecutando celdas individuales.

In [13]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

# 1. Verificar GPU y entorno Python
import sys
import subprocess

print(f"Ejecutable Python activo: {sys.executable}")
print(f"Versión de Python: {sys.version}")
print(f"📁 Raíz del proyecto (PROJECT_ROOT): {PROJECT_ROOT}")
print(f"📁 Carpeta de trabajo (MODELS_WORK_DIR): {MODELS_WORK_DIR}\n")

res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print("⚠️ nvidia-smi no disponible.")

Ejecutable Python activo: c:\Users\juanj\Desktop\SemaforoIA\.venv\Scripts\python.exe
Versión de Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
📁 Raíz del proyecto (PROJECT_ROOT): C:\Users\juanj\Desktop\SemaforoIA
📁 Carpeta de trabajo (MODELS_WORK_DIR): C:\Users\juanj\Desktop\SemaforoIA\models

Tue Nov 10 17:08:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 617.14                 KMD Version: 617.14        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+===========

## 2. Configuración Local de TensorFlow 2 Object Detection API

In [14]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import os
import sys
import shutil
from unittest.mock import MagicMock
from importlib.machinery import ModuleSpec

TF_MODELS_REPO = MODELS_WORK_DIR / 'models'
RESEARCH_DIR = TF_MODELS_REPO / 'research'
SLIM_DIR = RESEARCH_DIR / 'slim'

if not RESEARCH_DIR.exists():
    import subprocess
    print("Clonando repositorio oficial tensorflow/models...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tensorflow/models.git", str(TF_MODELS_REPO)], check=True)

repo_protos = RESEARCH_DIR / 'object_detection' / 'protos'
anchor_pb2 = repo_protos / 'anchor_generator_pb2.py'
if not anchor_pb2.exists():
    import subprocess
    import shutil
    print("Compilando archivos Protocol Buffers (.proto)...")
    protoc_bin = shutil.which("protoc")
    if not protoc_bin and sys.platform.startswith("linux"):
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "protobuf-compiler"], check=False)
        protoc_bin = shutil.which("protoc")
    if protoc_bin:
        if sys.platform.startswith("win"):
            for pf in repo_protos.glob("*.proto"):
                subprocess.run([protoc_bin, f"--proto_path={RESEARCH_DIR}", f"--python_out={RESEARCH_DIR}", str(pf)], check=False)
        else:
            subprocess.run(f"{protoc_bin} object_detection/protos/*.proto --python_out=.", shell=True, cwd=str(RESEARCH_DIR), check=True)
    if not anchor_pb2.exists():
        try:
            import object_detection
            pkg_protos = Path(object_detection.__file__).parent / 'protos'
            if pkg_protos.exists():
                for pb2_file in pkg_protos.glob('*_pb2.py'):
                    target = repo_protos / pb2_file.name
                    if not target.exists():
                        shutil.copy2(pb2_file, target)
        except Exception:
            pass
    if anchor_pb2.exists():
        print("✅ Archivos protobuf (.proto) compilados exitosamente.")


for p in [str(TF_MODELS_REPO), str(RESEARCH_DIR), str(SLIM_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

class AutoMockFinder:
    def __init__(self, prefixes):
        self.prefixes = prefixes
    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + '.') for p in self.prefixes):
            spec = ModuleSpec(fullname, None)
            spec.loader = self
            return spec
        return None
    def create_module(self, spec):
        mod = MagicMock()
        mod.__name__ = spec.name
        mod.__file__ = '<mock>'
        mod.__loader__ = self
        mod.__package__ = spec.name.rpartition('.')[0]
        mod.__path__ = []
        return mod
    def exec_module(self, module):
        pass

if not any(cls.__class__.__name__ == 'AutoMockFinder' for cls in sys.meta_path):
    sys.meta_path.insert(0, AutoMockFinder(['official', 'tensorflow_io', 'tensorflow.compat.v1.estimator', 'seqeval', 'tensorboard']))

import tf_keras
sys.modules['keras'] = tf_keras
import tensorflow as tf
tf.keras = tf_keras

print(f"📁 Raíz del proyecto: {PROJECT_ROOT}")
print(f"📁 Carpeta de trabajo models: {MODELS_WORK_DIR}")
print("✅ TensorFlow Object Detection API configurada exitosamente en el entorno local.")

📁 Raíz del proyecto: C:\Users\juanj\Desktop\SemaforoIA
📁 Carpeta de trabajo models: C:\Users\juanj\Desktop\SemaforoIA\models
✅ TensorFlow Object Detection API configurada exitosamente en el entorno local.


In [15]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import os
import sys
import shutil
from unittest.mock import MagicMock
from importlib.machinery import ModuleSpec

TF_MODELS_REPO = MODELS_WORK_DIR / 'models'
RESEARCH_DIR = TF_MODELS_REPO / 'research'
SLIM_DIR = RESEARCH_DIR / 'slim'

if not RESEARCH_DIR.exists():
    import subprocess
    print("Clonando repositorio oficial tensorflow/models...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tensorflow/models.git", str(TF_MODELS_REPO)], check=True)

repo_protos = RESEARCH_DIR / 'object_detection' / 'protos'
anchor_pb2 = repo_protos / 'anchor_generator_pb2.py'
if not anchor_pb2.exists():
    import subprocess
    import shutil
    print("Compilando archivos Protocol Buffers (.proto)...")
    protoc_bin = shutil.which("protoc")
    if not protoc_bin and sys.platform.startswith("linux"):
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "protobuf-compiler"], check=False)
        protoc_bin = shutil.which("protoc")
    if protoc_bin:
        if sys.platform.startswith("win"):
            for pf in repo_protos.glob("*.proto"):
                subprocess.run([protoc_bin, f"--proto_path={RESEARCH_DIR}", f"--python_out={RESEARCH_DIR}", str(pf)], check=False)
        else:
            subprocess.run(f"{protoc_bin} object_detection/protos/*.proto --python_out=.", shell=True, cwd=str(RESEARCH_DIR), check=True)
    if not anchor_pb2.exists():
        try:
            import object_detection
            pkg_protos = Path(object_detection.__file__).parent / 'protos'
            if pkg_protos.exists():
                for pb2_file in pkg_protos.glob('*_pb2.py'):
                    target = repo_protos / pb2_file.name
                    if not target.exists():
                        shutil.copy2(pb2_file, target)
        except Exception:
            pass
    if anchor_pb2.exists():
        print("✅ Archivos protobuf (.proto) compilados exitosamente.")


for p in [str(TF_MODELS_REPO), str(RESEARCH_DIR), str(SLIM_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

class AutoMockFinder:
    def __init__(self, prefixes):
        self.prefixes = prefixes
    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + '.') for p in self.prefixes):
            spec = ModuleSpec(fullname, None)
            spec.loader = self
            return spec
        return None
    def create_module(self, spec):
        mod = MagicMock()
        mod.__name__ = spec.name
        mod.__file__ = '<mock>'
        mod.__loader__ = self
        mod.__package__ = spec.name.rpartition('.')[0]
        mod.__path__ = []
        return mod
    def exec_module(self, module):
        pass

if not any(cls.__class__.__name__ == 'AutoMockFinder' for cls in sys.meta_path):
    sys.meta_path.insert(0, AutoMockFinder(['official', 'tensorflow_io', 'tensorflow.compat.v1.estimator', 'seqeval', 'tensorboard']))

import tf_keras
sys.modules['keras'] = tf_keras
import tensorflow as tf
tf.keras = tf_keras

# Verificación rápida de constructores de la API
from object_detection.builders import model_builder
from object_detection.protos import pipeline_pb2
print("✅ Módulos 'model_builder' y 'pipeline_pb2' importados y verificados correctamente.")

✅ Módulos 'model_builder' y 'pipeline_pb2' importados y verificados correctamente.


## 3. Verificar o Cargar el Dataset (`imagenes/` y `annotations_xml/`)

In [25]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import zipfile

if IMAGES_DIR.exists() and ANNOTATIONS_DIR.exists():
    num_imgs = len([p for p in IMAGES_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
    num_xmls = len(list(ANNOTATIONS_DIR.glob('*.xml')))
    print("✅ Dataset local detectado directamente:")
    print(f"   - Imágenes en '{IMAGES_DIR}': {num_imgs}")
    print(f"   - Anotaciones XML en '{ANNOTATIONS_DIR}': {num_xmls}")
elif ZIP_PATH.exists():
    print(f"Descomprimiendo {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_ROOT)
    print("✅ Dataset descomprimido exitosamente.")
else:
    print("⚠️ No se encontraron las carpetas 'imagenes/' y 'annotations_xml/' ni 'dataset_semaforo.zip'.")

✅ Dataset local detectado directamente:
   - Imágenes en 'C:\Users\juanj\Desktop\SemaforoIA\imagenes': 2353
   - Anotaciones XML en 'C:\Users\juanj\Desktop\SemaforoIA\annotations_xml': 921


## 4. Crear el Label Map (`label_map.pbtxt`)

In [26]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

label_map_content = """item {
  id: 1
  name: 'car'
}
"""
LABEL_MAP_PATH.write_text(label_map_content, encoding='utf-8')
print(f"✅ label_map.pbtxt creado en: {LABEL_MAP_PATH}")

✅ label_map.pbtxt creado en: C:\Users\juanj\Desktop\SemaforoIA\models\training_output\label_map.pbtxt


## 5. Convertir anotaciones Pascal VOC XML a TFRecords (`train.record` y `val.record`)

In [27]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import io
import random
import xml.etree.ElementTree as ET
from PIL import Image
import tensorflow as tf

TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def int64_list_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=value))

def bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def bytes_list_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=value))

def float_list_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=value))

def xml_to_tf_example(xml_path, images_dir):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        filename = root.find('filename').text
        stem = Path(filename).stem
        found_imgs = [p for p in Path(images_dir).rglob(f"{stem}.*") if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]
        if not found_imgs:
            return None
        
        image_path = found_imgs[0]
        # Convertir siempre a RGB de 3 canales JPEG para evitar errores en imágenes PNG RGBA de 4 canales
        with Image.open(image_path) as image:
            if image.mode != 'RGB':
                image = image.convert('RGB')
            width, height = image.size
            buf = io.BytesIO()
            image.save(buf, format='JPEG', quality=95)
            encoded_jpg = buf.getvalue()
        
        xmins, xmaxs, ymins, ymaxs = [], [], [], []
        classes_text, classes = [], []
        
        for member in root.findall('object'):
            bndbox = member.find('bndbox')
            if bndbox is None:
                continue
            xmin = max(0.0, min(1.0, float(bndbox.find('xmin').text) / width))
            xmax = max(0.0, min(1.0, float(bndbox.find('xmax').text) / width))
            ymin = max(0.0, min(1.0, float(bndbox.find('ymin').text) / height))
            ymax = max(0.0, min(1.0, float(bndbox.find('ymax').text) / height))
            if xmax <= xmin or ymax <= ymin:
                continue
            
            xmins.append(xmin)
            xmaxs.append(xmax)
            ymins.append(ymin)
            ymaxs.append(ymax)
            classes_text.append(b'car')
            classes.append(1)
            
        if not xmins:
            return None
            
        tf_example = tf.train.Example(features=tf.train.Features(feature={
            'image/height': int64_feature(height),
            'image/width': int64_feature(width),
            'image/filename': bytes_feature(filename.encode('utf8')),
            'image/source_id': bytes_feature(filename.encode('utf8')),
            'image/encoded': bytes_feature(encoded_jpg),
            'image/format': bytes_feature(b'jpeg'),
            'image/object/bbox/xmin': float_list_feature(xmins),
            'image/object/bbox/xmax': float_list_feature(xmaxs),
            'image/object/bbox/ymin': float_list_feature(ymins),
            'image/object/bbox/ymax': float_list_feature(ymaxs),
            'image/object/class/text': bytes_list_feature(classes_text),
            'image/object/class/label': int64_list_feature(classes),
        }))
        return tf_example
    except Exception:
        return None

xml_files = sorted(list(ANNOTATIONS_DIR.glob('*.xml')))
print(f"Total XMLs encontrados en '{ANNOTATIONS_DIR}': {len(xml_files)}")

random.seed(42)
random.shuffle(xml_files)

split_idx = max(1, int(0.85 * len(xml_files)))
train_xmls = xml_files[:split_idx]
val_xmls = xml_files[split_idx:] if split_idx < len(xml_files) else xml_files[:1]

if TRAIN_RECORD_PATH.exists() and VAL_RECORD_PATH.exists() and TRAIN_RECORD_PATH.stat().st_size > 0:
    print(f"✅ Archivos TFRecord listos en '{TRAINING_OUTPUT_DIR}':")
    print(f"   - train.record ({TRAIN_RECORD_PATH.stat().st_size / (1024*1024):.2f} MB)")
    print(f"   - val.record   ({VAL_RECORD_PATH.stat().st_size / (1024*1024):.2f} MB)")
else:
    for record_path, files in [(TRAIN_RECORD_PATH, train_xmls), (VAL_RECORD_PATH, val_xmls)]:
        writer = tf.io.TFRecordWriter(str(record_path))
        count = 0
        for f in files:
            tf_example = xml_to_tf_example(f, IMAGES_DIR)
            if tf_example is not None:
                writer.write(tf_example.SerializeToString())
                count += 1
        writer.close()
        print(f"✅ {record_path.name} creado con {count} ejemplos.")

Total XMLs encontrados en 'C:\Users\juanj\Desktop\SemaforoIA\annotations_xml': 921
✅ Archivos TFRecord listos en 'C:\Users\juanj\Desktop\SemaforoIA\models\training_output':
   - train.record (55.28 MB)
   - val.record   (12.99 MB)


## 6. Verificar o Descargar Modelo Base Preentrenado (SSD MobileNet V2)

In [28]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import tarfile
import urllib.request

if (PRETRAINED_DIR / 'checkpoint' / 'ckpt-0.index').exists():
    print(f"✅ Modelo base preentrenado encontrado localmente en:\n   {PRETRAINED_DIR}")
else:
    PRETRAINED_DIR.parent.mkdir(parents=True, exist_ok=True)
    tar_path = PRETRAINED_DIR.parent / f'{MODEL_NAME}.tar.gz'
    url = f'http://download.tensorflow.org/models/object_detection/tf2/20200711/{MODEL_NAME}.tar.gz'
    print(f"Descargando {MODEL_NAME}...")
    urllib.request.urlretrieve(url, str(tar_path))
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=PRETRAINED_DIR.parent)
    print(f"✅ Modelo preentrenado extraído en: {PRETRAINED_DIR}")

✅ Modelo base preentrenado encontrado localmente en:
   C:\Users\juanj\Desktop\SemaforoIA\pretrained\ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8


## 7. Configuración de `pipeline.config` con `pipeline_pb2`

In [29]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import os
import sys
import shutil
from unittest.mock import MagicMock
from importlib.machinery import ModuleSpec

TF_MODELS_REPO = MODELS_WORK_DIR / 'models'
RESEARCH_DIR = TF_MODELS_REPO / 'research'
SLIM_DIR = RESEARCH_DIR / 'slim'

if not RESEARCH_DIR.exists():
    import subprocess
    print("Clonando repositorio oficial tensorflow/models...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tensorflow/models.git", str(TF_MODELS_REPO)], check=True)

repo_protos = RESEARCH_DIR / 'object_detection' / 'protos'
anchor_pb2 = repo_protos / 'anchor_generator_pb2.py'
if not anchor_pb2.exists():
    import subprocess
    import shutil
    print("Compilando archivos Protocol Buffers (.proto)...")
    protoc_bin = shutil.which("protoc")
    if not protoc_bin and sys.platform.startswith("linux"):
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "protobuf-compiler"], check=False)
        protoc_bin = shutil.which("protoc")
    if protoc_bin:
        if sys.platform.startswith("win"):
            for pf in repo_protos.glob("*.proto"):
                subprocess.run([protoc_bin, f"--proto_path={RESEARCH_DIR}", f"--python_out={RESEARCH_DIR}", str(pf)], check=False)
        else:
            subprocess.run(f"{protoc_bin} object_detection/protos/*.proto --python_out=.", shell=True, cwd=str(RESEARCH_DIR), check=True)
    if not anchor_pb2.exists():
        try:
            import object_detection
            pkg_protos = Path(object_detection.__file__).parent / 'protos'
            if pkg_protos.exists():
                for pb2_file in pkg_protos.glob('*_pb2.py'):
                    target = repo_protos / pb2_file.name
                    if not target.exists():
                        shutil.copy2(pb2_file, target)
        except Exception:
            pass
    if anchor_pb2.exists():
        print("✅ Archivos protobuf (.proto) compilados exitosamente.")


for p in [str(TF_MODELS_REPO), str(RESEARCH_DIR), str(SLIM_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

class AutoMockFinder:
    def __init__(self, prefixes):
        self.prefixes = prefixes
    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + '.') for p in self.prefixes):
            spec = ModuleSpec(fullname, None)
            spec.loader = self
            return spec
        return None
    def create_module(self, spec):
        mod = MagicMock()
        mod.__name__ = spec.name
        mod.__file__ = '<mock>'
        mod.__loader__ = self
        mod.__package__ = spec.name.rpartition('.')[0]
        mod.__path__ = []
        return mod
    def exec_module(self, module):
        pass

if not any(cls.__class__.__name__ == 'AutoMockFinder' for cls in sys.meta_path):
    sys.meta_path.insert(0, AutoMockFinder(['official', 'tensorflow_io', 'tensorflow.compat.v1.estimator', 'seqeval', 'tensorboard']))

import tf_keras
sys.modules['keras'] = tf_keras
import tensorflow as tf
tf.keras = tf_keras

from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_STEPS = 200
BATCH_SIZE = 8

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with tf.io.gfile.GFile(str(BASE_CONFIG_PATH), 'r') as f:
    proto_str = f.read()
    text_format.Merge(proto_str, pipeline_config)

# 1 sola clase: car (vehículo)
pipeline_config.model.ssd.num_classes = 1
pipeline_config.train_config.batch_size = BATCH_SIZE
pipeline_config.train_config.fine_tune_checkpoint = str(PRETRAINED_DIR / 'checkpoint' / 'ckpt-0').replace('\\', '/')
pipeline_config.train_config.fine_tune_checkpoint_type = 'detection'
pipeline_config.train_config.num_steps = NUM_STEPS

pipeline_config.train_input_reader.label_map_path = str(LABEL_MAP_PATH).replace('\\', '/')
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:] = [str(TRAIN_RECORD_PATH).replace('\\', '/')]

pipeline_config.eval_input_reader[0].label_map_path = str(LABEL_MAP_PATH).replace('\\', '/')
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [str(VAL_RECORD_PATH).replace('\\', '/')]

config_text = text_format.MessageToString(pipeline_config)
with tf.io.gfile.GFile(str(CUSTOM_CONFIG_PATH), 'wb') as f:
    f.write(config_text)

print(f"✅ pipeline.config configurado exitosamente en:\n   {CUSTOM_CONFIG_PATH}")
print(f"   - Batch Size: {BATCH_SIZE} | Pasos totales: {NUM_STEPS}")

✅ pipeline.config configurado exitosamente en:
   C:\Users\juanj\Desktop\SemaforoIA\models\training_output\pipeline.config
   - Batch Size: 8 | Pasos totales: 200


## 8. Preparar e Iniciar el Entrenamiento Local

In [30]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

# Generar el script local entrenar.py adaptado a Windows / .venv
runner_code = """import sys
import os
from pathlib import Path
from unittest.mock import MagicMock
from importlib.machinery import ModuleSpec

# 1. Interceptor universal para módulos secundarios no requeridos
class AutoMockFinder:
    def __init__(self, prefixes):
        self.prefixes = prefixes
    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + '.') for p in self.prefixes):
            spec = ModuleSpec(fullname, None)
            spec.loader = self
            return spec
        return None
    def create_module(self, spec):
        mod = MagicMock()
        mod.__name__ = spec.name
        mod.__file__ = '<mock>'
        mod.__loader__ = self
        mod.__package__ = spec.name.rpartition('.')[0]
        mod.__path__ = []
        return mod
    def exec_module(self, module):
        pass

sys.meta_path.insert(0, AutoMockFinder(['official', 'tensorflow_io', 'tensorflow.compat.v1.estimator', 'seqeval', 'tensorboard']))

# 2. Inyectar tf_keras como motor principal de Keras (Keras 2 nativo para TF Object Detection)
import tf_keras
sys.modules['keras'] = tf_keras
import tensorflow as tf
tf.keras = tf_keras

# Evitar TBNotInstalledError en entornos sin TensorBoard completo
for _fn in ('image', 'scalar', 'histogram', 'text', 'audio', 'write', 'flush'):
    setattr(tf.summary, _fn, lambda *args, **kwargs: True)
    setattr(tf.compat.v2.summary, _fn, lambda *args, **kwargs: True)

# 3. Compatibilidad para control_flow_ops en tf_slim (decodificación de imágenes)
from tensorflow.python.ops import control_flow_ops
control_flow_ops.case = tf.case
control_flow_ops.cond = tf.cond
control_flow_ops.while_loop = tf.while_loop

# 4. Inyectar rutas locales de TensorFlow Object Detection
BASE_DIR = Path(__file__).resolve().parent
RESEARCH_DIR = BASE_DIR / 'models' / 'research'
SLIM_DIR = RESEARCH_DIR / 'slim'
for p in [str(BASE_DIR / 'models'), str(RESEARCH_DIR), str(SLIM_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from object_detection import model_lib_v2
from object_detection import model_main_tf2
from absl import app

model_lib_v2.LOG_EVERY = 10
model_lib_v2.NUM_STEPS_PER_ITERATION = 10
_orig_train_loop = model_lib_v2.train_loop

def _patched_train_loop(*args, **kwargs):
    train_steps = kwargs.get('train_steps') or 100
    step_iter = min(10, max(1, int(train_steps)))
    kwargs['num_steps_per_iteration'] = step_iter
    kwargs['checkpoint_every_n'] = min(50, step_iter)
    return _orig_train_loop(*args, **kwargs)

model_lib_v2.train_loop = _patched_train_loop

if __name__ == '__main__':
    app.run(model_main_tf2.main)
"""

ENTRENAR_SCRIPT.write_text(runner_code, encoding='utf-8')
print(f"✅ Script '{ENTRENAR_SCRIPT}' preparado exitosamente para entorno local.")

✅ Script 'C:\Users\juanj\Desktop\SemaforoIA\models\entrenar.py' preparado exitosamente para entorno local.


In [31]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import sys
import subprocess

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(ENTRENAR_SCRIPT),
    f"--model_dir={CHECKPOINT_DIR}",
    f"--pipeline_config_path={CUSTOM_CONFIG_PATH}",
    f"--num_train_steps={NUM_STEPS}",
    "--alsologtostderr"
]

print("🚀 Iniciando entrenamiento...")
print("Comando:", " ".join(cmd))

process = subprocess.Popen(
    cmd,
    cwd=str(MODELS_WORK_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
if process.returncode == 0:
    print("\n✅ Entrenamiento finalizado exitosamente.")
else:
    print(f"\n⚠️ El proceso terminó con código {process.returncode}.")

🚀 Iniciando entrenamiento...
Comando: c:\Users\juanj\Desktop\SemaforoIA\.venv\Scripts\python.exe C:\Users\juanj\Desktop\SemaforoIA\models\entrenar.py --model_dir=C:\Users\juanj\Desktop\SemaforoIA\models\training_checkpoint --pipeline_config_path=C:\Users\juanj\Desktop\SemaforoIA\models\training_output\pipeline.config --num_train_steps=200 --alsologtostderr
I0000 00:00:1794350047.380645   22732 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794350049.052700   22732 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

I0000 00:00:1794350052.286969   22732 cpu_feature_guard.cc:227] This TensorFlo

## 9. Exportar el Modelo Entrenado a `SavedModel`

In [32]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

import sys
import shutil
import subprocess

exporter_code = """import sys
import os
from pathlib import Path
from unittest.mock import MagicMock
from importlib.machinery import ModuleSpec

class AutoMockFinder:
    def __init__(self, prefixes):
        self.prefixes = prefixes
    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + '.') for p in self.prefixes):
            spec = ModuleSpec(fullname, None)
            spec.loader = self
            return spec
        return None
    def create_module(self, spec):
        mod = MagicMock()
        mod.__name__ = spec.name
        mod.__file__ = '<mock>'
        mod.__loader__ = self
        mod.__package__ = spec.name.rpartition('.')[0]
        mod.__path__ = []
        return mod
    def exec_module(self, module):
        pass

sys.meta_path.insert(0, AutoMockFinder(['official', 'tensorflow_io', 'tensorflow.compat.v1.estimator', 'seqeval', 'tensorboard']))

import tf_keras
sys.modules['keras'] = tf_keras
import tensorflow as tf
tf.keras = tf_keras

from tensorflow.python.ops import control_flow_ops
control_flow_ops.case = tf.case
control_flow_ops.cond = tf.cond
control_flow_ops.while_loop = tf.while_loop

BASE_DIR = Path(__file__).resolve().parent
RESEARCH_DIR = BASE_DIR / 'models' / 'research'
SLIM_DIR = RESEARCH_DIR / 'slim'
for p in [str(BASE_DIR / 'models'), str(RESEARCH_DIR), str(SLIM_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from object_detection import exporter_main_v2
from absl import app

if __name__ == '__main__':
    app.run(exporter_main_v2.main)
"""

EXPORTAR_SCRIPT.write_text(exporter_code, encoding='utf-8')

export_cmd = [
    sys.executable,
    str(EXPORTAR_SCRIPT),
    "--input_type=image_tensor",
    f"--pipeline_config_path={CUSTOM_CONFIG_PATH}",
    f"--trained_checkpoint_dir={CHECKPOINT_DIR}",
    f"--output_directory={EXPORTED_DIR}"
]

print("📦 Exportando SavedModel...")
res = subprocess.run(export_cmd, cwd=str(MODELS_WORK_DIR), capture_output=True, text=True)
print(res.stdout)
if res.returncode == 0:
    exported_sm = EXPORTED_DIR / 'saved_model'
    if exported_sm.exists():
        if FINAL_SAVED_MODEL_DIR.exists():
            shutil.rmtree(FINAL_SAVED_MODEL_DIR)
        shutil.copytree(exported_sm, FINAL_SAVED_MODEL_DIR)
        print(f"✅ Modelo exportado y actualizado directamente en: {FINAL_SAVED_MODEL_DIR}")
else:
    print(res.stderr)

📦 Exportando SavedModel...

✅ Modelo exportado y actualizado directamente en: C:\Users\juanj\Desktop\SemaforoIA\models\saved_model


## 10. Verificación Final del `SavedModel`

In [33]:
from pathlib import Path

# Resolver rutas del proyecto de forma autónoma en cualquier celda
_CWD = Path.cwd().resolve()
if _CWD.name == 'models' and (_CWD.parent / 'imagenes').exists():
    PROJECT_ROOT = _CWD.parent
    MODELS_WORK_DIR = _CWD
elif (_CWD / 'imagenes').exists() and (_CWD / 'models').exists():
    PROJECT_ROOT = _CWD
    MODELS_WORK_DIR = _CWD / 'models'
elif (Path.home() / 'Desktop' / 'SemaforoIA').exists():
    PROJECT_ROOT = Path.home() / 'Desktop' / 'SemaforoIA'
    MODELS_WORK_DIR = PROJECT_ROOT / 'models'
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == 'models' else _CWD
    MODELS_WORK_DIR = _CWD if _CWD.name == 'models' else (_CWD / 'models' if (_CWD / 'models').exists() else _CWD)

IMAGES_DIR = PROJECT_ROOT / 'imagenes'
ANNOTATIONS_DIR = PROJECT_ROOT / 'annotations_xml'
ZIP_PATH = PROJECT_ROOT / 'dataset_semaforo.zip'
TRAINING_OUTPUT_DIR = MODELS_WORK_DIR / 'training_output'
LABEL_MAP_PATH = TRAINING_OUTPUT_DIR / 'label_map.pbtxt'
TRAIN_RECORD_PATH = TRAINING_OUTPUT_DIR / 'train.record'
VAL_RECORD_PATH = TRAINING_OUTPUT_DIR / 'val.record'
MODEL_NAME = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_DIR = PROJECT_ROOT / 'pretrained' / MODEL_NAME
BASE_CONFIG_PATH = PRETRAINED_DIR / 'pipeline.config'
CUSTOM_CONFIG_PATH = TRAINING_OUTPUT_DIR / 'pipeline.config'
CHECKPOINT_DIR = MODELS_WORK_DIR / 'training_checkpoint'
ENTRENAR_SCRIPT = MODELS_WORK_DIR / 'entrenar.py'
EXPORTAR_SCRIPT = MODELS_WORK_DIR / 'exportar.py'
EXPORTED_DIR = MODELS_WORK_DIR / 'exported_model'
FINAL_SAVED_MODEL_DIR = MODELS_WORK_DIR / 'saved_model'
NUM_STEPS = globals().get('NUM_STEPS', 200)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)

saved_pb = FINAL_SAVED_MODEL_DIR / 'saved_model.pb'
if saved_pb.exists():
    print(f"✅ SavedModel listo para usar en modelo.ipynb: {saved_pb} ({saved_pb.stat().st_size / (1024*1024):.2f} MB)")
else:
    print("⚠️ Aún no se encuentra saved_model.pb en la carpeta de destino.")

✅ SavedModel listo para usar en modelo.ipynb: C:\Users\juanj\Desktop\SemaforoIA\models\saved_model\saved_model.pb (5.49 MB)
